# ============================================================
# 1. Определение цели, критериев и альтернатив
# ============================================================

In [24]:
import numpy as np
goal = "Выбор СУБД для нового микросервиса"

alternatives = [
    "PostgreSQL",
    "MongoDB",
    "MySQL",
]

criteria = [
    "Производительность",
    "Стоимость",
    "Масштабируемость",
    "Простота поддержки"
]

print(f"Метод анализа иерархий (AHP)")
print(f"Цель: {goal}")
print(f"Критерии: {', '.join(criteria)}")
print(f"Альтернативы: {', '.join(alternatives)}")



Метод анализа иерархий (AHP)
Цель: Выбор СУБД для нового микросервиса
Критерии: Производительность, Стоимость, Масштабируемость, Простота поддержки
Альтернативы: PostgreSQL, MongoDB, MySQL


# ============================================================
# 2. Шкала Саати
# ============================================================

In [25]:
def print_saaty_scale():
    print("\nШкала относительной важности Саати:")
    print("1 - Одинаковая важность")
    print("2 - Равномерная важность (промежуточное)")
    print("3 - Умеренное превосходство")
    print("4 - Существенное превосходство (промежуточное)")
    print("5 - Значительное превосходство")
    print("6 - Очень сильное превосходство (промежуточное)")
    print("7 - Очень сильное превосходство")
    print("8 - Превосходство высокой степени (промежуточное)")
    print("9 - Абсолютное превосходство")
    print("Обратные значения: 1/2, 1/3, ..., 1/9")

# ============================================================
# 3. Ввод матрицы с клавиатуры
# ============================================================

In [26]:

def input_matrix(size, name, is_pairwise=True):
    """
    Ввод матрицы парных сравнений размера size x size
    is_pairwise: True - заполняется только верхний треугольник (a_ij = 1/a_ji)
    """
    print(f"\nВвод матрицы парных сравнений для: {name}")
    print(f"Размер матрицы: {size}x{size}")
    print_saaty_scale()
    
    matrix = np.ones((size, size))
    
    for i in range(size):
        for j in range(i+1, size):
            while True:
                try:
                    if is_pairwise:
                        val = input(f"  Элемент [{i+1},{j+1}] (важность {i+1} относительно {j+1}): ")
                        val = float(val.replace('/', ' ').split()[0] if '/' in val else val)
                        # val - это a_ij (насколько i важнее j)
                    else:
                        val = float(input(f"  Элемент [{i+1},{j+1}]: "))
                    
                    if val <= 0:
                        print("Значение должно быть положительным! Попробуйте снова.")
                        continue
                    
                    matrix[i, j] = val
                    matrix[j, i] = 1.0 / val
                    break
                except ValueError:
                    print("Ошибка ввода! Введите число (например: 1, 3, 5, 7, 9 или 1/3, 1/5, 1/7, 1/9)")
                    continue
    
    # Диагональные элементы = 1
    for i in range(size):
        matrix[i, i] = 1.0
    
    return matrix


# ============================================================
# 4. Расчёт вектора приоритетов (геометрическое среднее -> нормализация)
# ============================================================

In [27]:

def calculate_priority_vector(matrix):
    """
    Расчёт вектора приоритетов методом геометрического среднего
    (более стабильный, чем собственный вектор)
    """
    n = matrix.shape[0]
    # Геометрическое среднее по строкам
    geo_means = np.exp(np.mean(np.log(matrix + 1e-10), axis=1))
    # Нормализация
    priority_vector = geo_means / np.sum(geo_means)
    return priority_vector

# ============================================================
# 5. Проверка согласованности (CI и CR)
# ============================================================

In [28]:

def check_consistency(matrix, priority_vector):
    """
    Вычисление лямбда_max, CI (индекс согласованности) и CR (отношение согласованности)
    """
    n = matrix.shape[0]
    # Вычисляем lambda_max = среднее (A * w) / w
    Aw = matrix @ priority_vector
    lambda_max = np.mean(Aw / (priority_vector + 1e-10))
    
    # Индекс согласованности
    CI = (lambda_max - n) / (n - 1)
    
    # Случайный индекс для n (по таблице Саати)
    RI_values = {1:0.0, 2:0.0, 3:0.58, 4:0.9, 5:1.12, 6:1.24, 7:1.32, 8:1.41, 9:1.45, 10:1.49}
    RI = RI_values.get(n, 1.49)
    
    # Отношение согласованности
    CR = CI / RI if RI > 0 else 0.0
    
    return lambda_max, CI, CR

# ============================================================
# 6. Ввод матрицы с проверкой CR
# ============================================================

In [29]:

def input_matrix_with_consistency(size, name):
    max_attempts = 5
    for attempt in range(max_attempts):
        matrix = input_matrix(size, name)
        pv = calculate_priority_vector(matrix)
        lambda_max, CI, CR = check_consistency(matrix, pv)
        
        print(f"\nРезультаты проверки согласованности для {name}:")
        print(f"  lambda_max = {lambda_max:.4f}")
        print(f"  CI = {CI:.4f}")
        print(f"  CR = {CR:.4f}")
        
        if CR <= 0.1:
            print(f" Согласованность приемлема (CR <= 0.1)")
            return matrix, pv
        else:
            print(f" Согласованность НЕПРИЕМЛЕМА (CR > 0.1)")
            if attempt < max_attempts - 1:
                print(f"  Пожалуйста, повторите ввод матрицы (попытка {attempt + 2}/{max_attempts})")
            else:
                print(f"  Достигнуто максимальное число попыток. Используется последняя матрица.")
                return matrix, pv
    
    return matrix, pv



# ============================================================
# 7. Основной расчёт AHP
# ============================================================


In [30]:
criteria_matrix, criteria_weights = input_matrix_with_consistency(len(criteria), "КРИТЕРИИ (относительно цели)")

# Простой вывод матрицы критериев
print("\n" + "=" * 70)
print("МАТРИЦА КРИТЕРИЕВ:")
for i in range(len(criteria)):
    row = ""
    for j in range(len(criteria)):
        row += f"{criteria_matrix[i][j]:>8.3f} "
    print(f"  {row}")
print()

print("Вектор приоритетов критериев:")
for i, crit in enumerate(criteria):
    print(f"  {crit}: {criteria_weights[i]:.4f} ({criteria_weights[i]*100:.2f}%)")

alt_matrices = []
alt_weights = []

for crit_idx, crit_name in enumerate(criteria):
    print("\n" + "=" * 70)
    matrix, pv = input_matrix_with_consistency(len(alternatives), f"АЛЬТЕРНАТИВЫ относительно критерия '{crit_name}'")
    alt_matrices.append(matrix)
    alt_weights.append(pv)
    
    # Простой вывод матрицы альтернатив
    print(f"\nМАТРИЦА АЛЬТЕРНАТИВ ДЛЯ КРИТЕРИЯ '{crit_name}':")
    for i in range(len(alternatives)):
        row = ""
        for j in range(len(alternatives)):
            val = matrix[i][j]
            if val < 1:
                row += f"{val:>8.3f} "
            else:
                row += f"{val:>8.2f} "
        print(f"  {row}")
    
    print(f"\nВектор приоритетов альтернатив для критерия '{crit_name}':")
    for i, alt in enumerate(alternatives):
        print(f"  {alt}: {pv[i]:.4f} ({pv[i]*100:.2f}%)")


Ввод матрицы парных сравнений для: КРИТЕРИИ (относительно цели)
Размер матрицы: 4x4

Шкала относительной важности Саати:
1 - Одинаковая важность
2 - Равномерная важность (промежуточное)
3 - Умеренное превосходство
4 - Существенное превосходство (промежуточное)
5 - Значительное превосходство
6 - Очень сильное превосходство (промежуточное)
7 - Очень сильное превосходство
8 - Превосходство высокой степени (промежуточное)
9 - Абсолютное превосходство
Обратные значения: 1/2, 1/3, ..., 1/9

Результаты проверки согласованности для КРИТЕРИИ (относительно цели):
  lambda_max = 4.1686
  CI = 0.0562
  CR = 0.0624
 Согласованность приемлема (CR <= 0.1)

МАТРИЦА КРИТЕРИЕВ:
     1.000    1.000    2.000    3.000 
     1.000    1.000    1.000    4.000 
     0.500    1.000    1.000    1.000 
     0.333    0.250    1.000    1.000 

Вектор приоритетов критериев:
  Производительность: 0.3592 (35.92%)
  Стоимость: 0.3245 (32.45%)
  Масштабируемость: 0.1930 (19.30%)
  Простота поддержки: 0.1233 (12.33%)





# ============================================================
# 8. Расчёт глобальных приоритетов и вывод результатов
# ============================================================

In [31]:

global_priorities = np.zeros(len(alternatives))

for crit_idx, crit_weight in enumerate(criteria_weights):
    global_priorities += crit_weight * alt_weights[crit_idx]

print("\n" + "=" * 70)
print("ИТОГОВЫЙ РЕЙТИНГ АЛЬТЕРНАТИВ")
print("=" * 70)
print("\n{:<15} {:>20}".format("Альтернатива", "Глобальный приоритет"))
print("-" * 40)

sorted_indices = np.argsort(global_priorities)[::-1]
for rank, idx in enumerate(sorted_indices, 1):
    alt = alternatives[idx]
    priority = global_priorities[idx]
    print(f"{rank}. {alt:<15} {priority:>20.4f} ({priority*100:.2f}%)")

best_alt = alternatives[sorted_indices[0]]
best_priority = global_priorities[sorted_indices[0]]

print("\n" + "=" * 70)
print(f"РЕКОМЕНДАЦИЯ: Оптимальная СУБД - {best_alt}")
print(f"Глобальный приоритет: {best_priority:.4f} ({best_priority*100:.2f}%)")
print("=" * 70)

# Дополнительный вывод: сводная таблица локальных весов
print("\nСводная таблица локальных весов альтернатив по критериям:")
print("\n{:<15}".format("Альтернатива"), end="")
for crit in criteria:
    print(f"{crit:>20}", end="")
print(f"{'Глобальный':>20}")

print("-" * 75)
for i, alt in enumerate(alternatives):
    print(f"{alt:<15}", end="")
    for crit_idx in range(len(criteria)):
        print(f"{alt_weights[crit_idx][i]:>20.4f}", end="")
    print(f"{global_priorities[i]:>20.4f}")


ИТОГОВЫЙ РЕЙТИНГ АЛЬТЕРНАТИВ

Альтернатива    Глобальный приоритет
----------------------------------------
1. PostgreSQL                    0.5042 (50.42%)
2. MongoDB                       0.2780 (27.80%)
3. MySQL                         0.2178 (21.78%)

РЕКОМЕНДАЦИЯ: Оптимальная СУБД - PostgreSQL
Глобальный приоритет: 0.5042 (50.42%)

Сводная таблица локальных весов альтернатив по критериям:

Альтернатива     Производительность           Стоимость    Масштабируемость  Простота поддержки          Глобальный
---------------------------------------------------------------------------
PostgreSQL                   0.5842              0.5499              0.3874              0.3333              0.5042
MongoDB                      0.2318              0.2098              0.4434              0.3333              0.2780
MySQL                        0.1840              0.2402              0.1692              0.3333              0.2178
